In [1]:
import json
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
reg = []
for file in Path("../bronze/data/records").glob("*.jsonl"):
    with open(file, encoding="utf-8") as f:
        for line in f:
            payload = json.loads(line)
            reg.extend(payload["results"])

In [3]:
df = pd.DataFrame(pd.json_normalize(reg))

In [4]:
raw_lines, raw_columns = df.shape

In [5]:
df = df.drop_duplicates(subset="id", keep="last")

In [6]:
df.head()

""


In [7]:
new_lines, new_columns = df.shape

print(f"{raw_lines - new_lines} lines were deleted")

0 lines were deleted


In [8]:
df = df.drop(columns = ["adref", "__CLASS__", "company.__CLASS__", "category.label", "category.tag", "location.area", "category.__CLASS__", "location.__CLASS__", "latitude","longitude"], errors='ignore')

In [9]:
df = df.rename(columns={'created':"collected_at", 'redirect_url':"url", 'salary_is_predicted':"salary", 'company.display_name':"company",'location.area':"local", 'location.display_name':"location"})

In [10]:
df = df.drop_duplicates()

In [11]:
df.to_parquet('data/data.parquet')